# Session 03 Practice: Road Network Graphs & Dijkstra's Shortest Path Search
**Course:** Motion Planning for Autonomous Vehicles (Planning P2)  
**Week 2 | Session 3** (Instructor Reference Solution)

---

### 🎯 Learning Objectives
1. Construct directed road network graphs $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ with spatial node coordinates and speed limits.
2. Implement **Dijkstra's Shortest Path Algorithm** from scratch using a min-heap priority queue (`heapq`).
3. Formulate edge travel time weights: $w(u, v) = \frac{d(u, v)}{v_{\text{speed}}}$ and multi-objective routing costs.
4. Visualize road networks, explored frontiers, and optimal route telemetry using **Plotly**.
5. Acquire, enrich, and model real-world street networks using **OSMnx** for autonomous vehicle mission planning.
6. Benchmark route optimization trade-offs between **Shortest Distance** vs **Fastest Travel Time** policies on complex urban multi-lane topologies.
7. Observe and analyze how the **Priority Queue min-heap** prunes unvisited search space via early termination.

---

### 📐 Mathematical Formulation

Given a directed graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$:
- **Cost-to-come:** $g(v) = \min_{u \in \text{pred}(v)} \left( g(u) + w(u, v) \right)$
- **Edge Weight (Travel Time):** $w_{\text{time}}(u, v) = \frac{d(u, v)}{v_{\text{limit}}(u, v)}$
- **Edge Weight (Distance):** $w_{\text{dist}}(u, v) = d(u, v) = \sqrt{(x_v - x_u)^2 + (y_v - y_u)^2}$
- **Priority Queue Ordering Key:** $\text{Key}(v) = g(v)$


In [ ]:
import heapq
import math
import os
import networkx as nx
import osmnx as ox
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, HTML, Video, Image
from shapely import wkt

# Set default plotly template for clean visualization
pio.templates.default = "plotly_white"


# Part 1: Reference Example — Synthetic Grid Road Network

In this first part, we demonstrate how mission planning graphs are constructed from geometric specifications and how Dijkstra's algorithm finds the global minimum-time path. This serves as an end-to-end reference example.

---

## 1. Road Network Construction

We define a 9-intersection city block graph with spatial $(x, y)$ coordinates (in meters) and speed limits (50 km/h $\approx 13.8$ m/s, 80 km/h $\approx 22.2$ m/s).


In [ ]:
def create_sample_road_network() -> nx.DiGraph:
    G = nx.DiGraph()
    
    nodes = {
        0: (0.0, 0.0),
        1: (100.0, 0.0),
        2: (250.0, 0.0),
        3: (0.0, 100.0),
        4: (100.0, 100.0),
        5: (250.0, 100.0),
        6: (0.0, 250.0),
        7: (120.0, 250.0),
        8: (250.0, 250.0),
    }
    for node_id, pos in nodes.items():
        G.add_node(node_id, pos=pos)
        
    # Directed road segments (u, v, speed_limit_m/s)
    # 50 km/h ~ 13.8 m/s, 80 km/h ~ 22.2 m/s
    edges = [
        (0, 1, 13.8), (1, 2, 22.2), (1, 0, 13.8), (2, 1, 22.2),
        (3, 4, 13.8), (4, 5, 13.8), (4, 3, 13.8), (5, 4, 13.8),
        (6, 7, 13.8), (7, 8, 22.2), (7, 6, 13.8), (8, 7, 22.2),
        (0, 3, 13.8), (3, 6, 13.8), (3, 0, 13.8), (6, 3, 13.8),
        (1, 4, 13.8), (4, 7, 13.8), (4, 1, 13.8), (7, 4, 13.8),
        (2, 5, 22.2), (5, 8, 22.2), (5, 2, 22.2), (8, 5, 22.2),
        (4, 8, 13.8), # diagonal avenue
    ]
    
    for u, v, speed in edges:
        pos_u = nodes[u]
        pos_v = nodes[v]
        dist = math.hypot(pos_v[0] - pos_u[0], pos_v[1] - pos_u[1])
        travel_time = dist / speed
        G.add_edge(u, v, length=dist, speed=speed, weight=travel_time)
        
    return G

G = create_sample_road_network()
print(f"Road network created: {len(G.nodes)} intersections, {len(G.edges)} directed road segments.")


### 🛠️ Tutorial: Creating and Accessing Graph Components in NetworkX

Before implementing shortest path algorithms, let us review how autonomous driving road networks are represented and traversed using `networkx`:

#### 1. Graph Types
- **`nx.DiGraph` (Standard Directed Graph):** Represents one-way or two-way roads where at most **one** directed edge exists from intersection $u$ to intersection $v$.
- **`nx.MultiDiGraph` (Multi Directed Graph):** Used by **OpenStreetMap / OSMnx**. It allows **multiple parallel edges** between the same pair of intersections (e.g., separate lanes, express lanes, or distinct road segments with different speed limits).

#### 2. Creating Nodes and Attaching Spatial Coordinates
```python
import networkx as nx

# Instantiate a directed graph
G = nx.DiGraph()

# Add nodes with custom spatial attributes (e.g. coordinates in meters or lat/lon)
G.add_node(0, pos=(0.0, 0.0))
G.add_node(1, pos=(100.0, 0.0))

# Access node attributes:
coord = G.nodes[0]['pos']     # Returns (0.0, 0.0)
all_nodes = list(G.nodes)     # [0, 1]
```

#### 3. Creating Directed Edges with Routing Attributes
```python
# Add directed edges: G.add_edge(u, v, **attributes)
G.add_edge(0, 1, length=100.0, speed=13.8, weight=7.25)
G.add_edge(1, 0, length=100.0, speed=13.8, weight=7.25)

# Access edge attributes in a DiGraph:
edge_dict = G[0][1]           # {'length': 100.0, 'speed': 13.8, 'weight': 7.25}
cost = G[0][1]['weight']      # 7.25
```

#### 4. Navigating Graph Neighbors
```python
# Outgoing neighbors (edges leaving u toward v):
for v in G.neighbors(0):
    print(f"Road leads from 0 to {v} with cost {G[0][v]['weight']}")

# Incoming predecessors (edges entering v from u):
for u in G.predecessors(1):
    print(f"Road arrives at 1 from {u}")
```

#### 5. Handling Parallel Edges in `nx.MultiDiGraph` (OSMnx)
In OpenStreetMap (`MultiDiGraph`), between two intersections $u$ and $v$ there can be parallel edges. NetworkX stores them as a nested dictionary keyed by edge index: `{0: attr_dict_0, 1: attr_dict_1}`.
To safely extract the minimum weight among all parallel connections:
```python
edge_dict = graph[u][v]
if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
    w = min(float(d.get(weight, float('inf'))) for d in edge_dict.values())
else:
    w = float(edge_dict.get(weight, float('inf')))
```

---

## 2. Dijkstra Algorithm Implementation (Pseudocode & Reference)

Dijkstra's algorithm systematically solves the Single-Source Shortest Path (SSSP) problem using a binary min-heap (`heapq`). Rather than stopping early at an intermediate node, classic Dijkstra explores outwards, settling all reachable nodes in order of minimum cumulative cost $g(u)$ and constructing the full shortest-path tree.

---

### 📝 Algorithm: Dijkstra Shortest Path (Priority Queue without Early Stop)

```text
Input:
    graph: Directed graph G = (V, E)
    start_node: Origin intersection node ID
    goal_node: Destination intersection node ID
    weight: Edge attribute to minimize (e.g. 'weight', 'travel_time', or 'length')

Output:
    path: List of node IDs along optimal route [start_node, ..., goal_node]
    cost: Total accumulated cost to goal_node
    num_explored: Total number of settled nodes
    explored_order: Sequence of node IDs in the order they were settled

1. Initialize g_score table:
     For every node in graph.nodes:
         g_score[node] = infinity
     g_score[start_node] = 0.0

2. Initialize parent pointer table:
     For every node in graph.nodes:
         parent[node] = None

3. Initialize priority queue and tracking sets:
     pq = [(0.0, start_node)]    # Min-heap ordered by current cumulative cost
     visited = empty set()       # Closed set of settled nodes
     explored_order = empty list # Chronological expansion history

4. While pq is not empty:
     a. Pop (current_g, u) with minimum current_g from pq
     b. If u is in visited:
          continue (skip stale/duplicate entries in the heap)
     c. Add u to visited
     d. Append u to explored_order
     e. (NO EARLY STOP: Continue exploration until pq is exhausted to build complete tree)
     f. For each outgoing neighbor v in graph.neighbors(u):
          - Extract edge weight w = cost of edge (u -> v)
          - Calculate tentative_g = current_g + w
          - If tentative_g < g_score[v]:
               g_score[v] = tentative_g
               parent[v] = u
               Push (tentative_g, v) onto pq

5. Reconstruct optimal path from goal_node back to start_node:
     If g_score[goal_node] == infinity:
         return ([], infinity, len(explored_order), explored_order)
     
     path = []
     curr = goal_node
     while curr is not None:
         path.append(curr)
         curr = parent[curr]
     path.reverse()
     return (path, g_score[goal_node], len(explored_order), explored_order)
```


In [ ]:
def dijkstra_shortest_path(graph: nx.DiGraph, start_node: int, goal_node: int) -> tuple[list[int], float, int, list[int]]:
    """Computes the shortest path using Dijkstra's algorithm with a binary min-heap priority queue (without early stop)."""
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    parent = {node: None for node in graph.nodes}
    
    pq = [(0.0, start_node)]
    visited = set()
    explored_order = []
    
    while pq:
        current_g, u = heapq.heappop(pq)
        
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        for v in graph.neighbors(u):
            weight = float(graph[u][v].get('weight', float('inf')))
            tentative_g = current_g + weight
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                heapq.heappush(pq, (tentative_g, v))
                
    if g_score[goal_node] == float('inf'):
        return [], float('inf'), len(explored_order), explored_order
        
    path = []
    curr = goal_node
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()
    return path, g_score[goal_node], len(explored_order), explored_order

path, cost, explored, order = dijkstra_shortest_path(G, 0, 8)
print(f"Optimal route from Node 0 to Node 8: {path}")
print(f"Total estimated travel time: {cost:.2f} seconds ({explored} nodes explored)")


## 3. Interactive Route Visualizer & Segment Telemetry Plots

In [ ]:
def plot_dijkstra_dashboard(graph: nx.DiGraph, path: list[int], explored_order: list[int]):
    edge_x = []
    edge_y = []
    for u, v in graph.edges():
        x0, y0 = graph.nodes[u]['pos']
        x1, y1 = graph.nodes[v]['pos']
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
    node_x = [graph.nodes[n]['pos'][0] for n in graph.nodes]
    node_y = [graph.nodes[n]['pos'][1] for n in graph.nodes]
    node_text = [f"Node {n}<br>Explored Step: {explored_order.index(n) if n in explored_order else 'Unvisited'}" for n in graph.nodes]
    
    fig = go.Figure()
    
    # 1. Base Road Edges
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=2, color='#b0bec5'),
        hoverinfo='none',
        name='Road Network'
    ))
    
    # 2. Intersections
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        marker=dict(size=14, color='#90caf9', line=dict(width=2, color='#1565c0')),
        text=[f"N{n}" for n in graph.nodes],
        textposition="top center",
        hovertext=node_text,
        hoverinfo="text",
        name='Intersections'
    ))
    
    # 3. Optimal Path
    if path:
        path_x = [graph.nodes[n]['pos'][0] for n in path]
        path_y = [graph.nodes[n]['pos'][1] for n in path]
        fig.add_trace(go.Scatter(
            x=path_x, y=path_y,
            mode='lines+markers',
            line=dict(width=6, color='#d32f2f'),
            marker=dict(size=16, color='#ffd600', symbol="star"),
            name='Dijkstra Optimal Route'
        ))
        
    fig.update_layout(
        title="Mission Planning: Road Network Graph & Dijkstra Optimal Shortest Path",
        xaxis_title="Global X (m)",
        yaxis_title="Global Y (m)",
        showlegend=True,
        width=850,
        height=600
    )
    fig.show()

plot_dijkstra_dashboard(G, path, order)


In [ ]:
# Segment Breakdown Plot (Time & Distance per Segment)
segment_names = []
segment_times = []
segment_lens = []

for i in range(len(path) - 1):
    u, v = path[i], path[i+1]
    segment_names.append(f"N{u} → N{v}")
    segment_times.append(G[u][v]["weight"])
    segment_lens.append(G[u][v]["length"])

fig_seg = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Segment Travel Time (seconds)", "Segment Distance (meters)")
)

fig_seg.add_trace(
    go.Bar(x=segment_names, y=segment_times, marker_color="#1f77b4", name="Travel Time (s)"),
    row=1, col=1
)
fig_seg.add_trace(
    go.Bar(x=segment_names, y=segment_lens, marker_color="#ff7f0e", name="Distance (m)"),
    row=1, col=2
)

fig_seg.update_layout(title="Optimal Route Segment Profile", height=380, showlegend=False)
fig_seg.show()


## 4. Verification Tests (Part 1)

In [ ]:
nx_path = nx.dijkstra_path(G, 0, 8, weight="weight")
nx_cost = nx.dijkstra_path_length(G, 0, 8, weight="weight")

assert path == nx_path, f"Computed path {path} does not match NetworkX {nx_path}!"
assert math.isclose(cost, nx_cost, rel_tol=1e-4), f"Computed cost {cost} does not match NetworkX {nx_cost}!"

print("✅ All synthetic grid Dijkstra tests passed successfully!")


---

# Part 2: Mission Planning on Real-World Urban Networks with OSMnx

In production autonomous vehicle software stacks (such as Waymo, Cruise, or Baidu Apollo), mission planners do not navigate toy grid graphs. Instead, they ingest high-definition map topologies and OpenStreetMap (OSM) road networks containing:
- Thousands of intersections with real GPS coordinates (latitude / longitude).
- Multi-lane, one-way, and divided roads.
- Varied road classifications (`motorway`, `primary`, `secondary`, `tertiary`, `residential`) with distinct statutory speed limits.
- Directed multi-edges (`MultiDiGraph`) where parallel links can connect the same pair of intersections with different lane properties.

In this practical exercise, you will apply Dijkstra's algorithm to plan autonomous driving missions on the street network of **Piedmont, California** and analyze the fundamental trade-off between **Distance-Optimal Routing** and **Time-Optimal Routing**.

---

## 5. OpenStreetMap Data Acquisition & Preprocessing

We use `osmnx` to load and prepare a drivable street graph. We calculate realistic edge traversal speeds and travel times:
$$v_{\text{limit}} = \text{speed\_kph} \times \frac{1000}{3600} \text{ m/s}$$
$$\text{travel\_time} = \frac{\text{length}}{v_{\text{limit}}}$$


In [ ]:
# Load or download the road network
data_dir = "../data"
graph_path = os.path.join(data_dir, "piedmont_road_network.graphml")

if os.path.exists(graph_path):
    print(f"Loading cached road network from: {graph_path}")
    G_osm = ox.load_graphml(filepath=graph_path)
else:
    print("Fetching OpenStreetMap network for Piedmont, CA...")
    G_osm = ox.graph_from_place("Piedmont, California, USA", network_type="drive")
    G_osm = ox.add_edge_speeds(G_osm)
    G_osm = ox.add_edge_travel_times(G_osm)
    os.makedirs(data_dir, exist_ok=True)
    ox.save_graphml(G_osm, filepath=graph_path)
    print(f"Saved road network to {graph_path}")

print(f"Urban Network Loaded: {len(G_osm.nodes)} intersections, {len(G_osm.edges)} directed road links.")

# Convert edge attributes to float to guarantee precision
for u, v, k, data in G_osm.edges(keys=True, data=True):
    data['length'] = float(data.get('length', 1.0))
    data['travel_time'] = float(data.get('travel_time', data['length'] / 13.8))


In [ ]:
# Inspect sample edge attributes using Polars
sample_edges = []
for u, v, k, data in list(G_osm.edges(keys=True, data=True))[:5]:
    sample_edges.append({
        "Origin": u,
        "Destination": v,
        "Highway": str(data.get("highway", "unclassified")),
        "Length (m)": round(float(data.get("length", 0.0)), 2),
        "Speed Limit (km/h)": float(data.get("speed_kph", 40.0)),
        "Travel Time (s)": round(float(data.get("travel_time", 0.0)), 2)
    })

df_sample = pl.DataFrame(sample_edges)
print(df_sample)


## 6. Real-World Multi-Attribute Dijkstra Planner (Reference Implementation)

Below is the complete reference implementation of `osmnx_dijkstra`, supporting both directed `MultiDiGraph` and `DiGraph` structures with arbitrary edge weighting attributes (`length` or `travel_time`).


In [ ]:
def osmnx_dijkstra(
    graph: nx.MultiDiGraph, 
    start_node: int, 
    goal_node: int, 
    weight: str = 'travel_time'
) -> tuple[list[int], float, int, list[int]]:
    """Executes Dijkstra's algorithm on an OSMnx MultiDiGraph using a priority queue (without early stop)."""
    g_score = {node: float('inf') for node in graph.nodes}
    g_score[start_node] = 0.0
    parent = {node: None for node in graph.nodes}
    
    pq = [(0.0, start_node)]
    visited = set()
    explored_order = []
    
    while pq:
        current_g, u = heapq.heappop(pq)
        
        if u in visited:
            continue
        visited.add(u)
        explored_order.append(u)
        
        for v in graph.neighbors(u):
            edge_data = graph[u][v]
            if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
                w = min(float(d.get(weight, float('inf'))) for d in edge_data.values())
            else:
                w = float(edge_data.get(weight, float('inf')))
                
            tentative_g = current_g + w
            if tentative_g < g_score[v]:
                g_score[v] = tentative_g
                parent[v] = u
                heapq.heappush(pq, (tentative_g, v))
                
    if g_score[goal_node] == float('inf'):
        return [], float('inf'), len(explored_order), explored_order
        
    path = []
    curr = goal_node
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()
    return path, g_score[goal_node], len(explored_order), explored_order


## 7. Autonomous Mission Benchmark: Shortest Distance vs Fastest Travel Time

In autonomous fleet operations (e.g. robotaxis), dispatch systems evaluate routes under different objectives:
1. **Shortest Distance Policy ($\min \sum \text{length}$):** Minimizes vehicle wear and odometer accumulation. However, it often guides the vehicle through narrow, speed-limited residential streets with frequent stop signs.
2. **Fastest Travel Time Policy ($\min \sum \text{travel\_time}$):** Prioritizes wider arterials and primary corridors with higher speed limits, achieving significantly faster transit despite a slightly longer physical distance.

Let's compute and compare both routes between two distant city coordinates.


In [ ]:
# Select two connected nodes spanning the urban corridor
all_nodes = list(G_osm.nodes)
start_node = all_nodes[10]
reachable_nodes = [n for n in all_nodes if n != start_node and nx.has_path(G_osm, start_node, n)]
goal_node = reachable_nodes[-15]

print(f"Origin Node ID:      {start_node} (Lat: {G_osm.nodes[start_node]['y']:.5f}, Lon: {G_osm.nodes[start_node]['x']:.5f})")
print(f"Destination Node ID: {goal_node} (Lat: {G_osm.nodes[goal_node]['y']:.5f}, Lon: {G_osm.nodes[goal_node]['x']:.5f})")

# 1. Run Dijkstra for Fastest Travel Time Route
path_fastest, time_fastest, exp_fastest, order_fastest = osmnx_dijkstra(
    G_osm, start_node, goal_node, weight='travel_time'
)

# 2. Run Dijkstra for Shortest Distance Route
path_shortest, dist_shortest, exp_shortest, order_shortest = osmnx_dijkstra(
    G_osm, start_node, goal_node, weight='length'
)

# Function to compute cross-metrics along any node path
def compute_path_metrics(graph: nx.MultiDiGraph, path: list[int]) -> tuple[float, float]:
    total_dist = 0.0
    total_time = 0.0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]
        edge_data = graph[u][v]
        if isinstance(graph, (nx.MultiDiGraph, nx.MultiGraph)):
            min_tt = float('inf')
            best_len = 0.0
            for d in edge_data.values():
                tt = float(d.get('travel_time', float('inf')))
                if tt < min_tt:
                    min_tt = tt
                    best_len = float(d.get('length', 0.0))
            total_dist += best_len
            total_time += min_tt
        else:
            total_dist += float(edge_data.get('length', 0.0))
            total_time += float(edge_data.get('travel_time', 0.0))
    return total_dist, total_time

dist_fastest, _ = compute_path_metrics(G_osm, path_fastest)
_, time_shortest = compute_path_metrics(G_osm, path_shortest)

# Tabular comparison with Polars
comparison_df = pl.DataFrame({
    "Mission Telemetry Metric": [
        "Total Path Distance (meters)",
        "Total Estimated Travel Time (seconds)",
        "Total Estimated Travel Time (minutes)",
        "Average Vehicle Speed (km/h)",
        "Waypoints in Path (Nodes)",
        "Search Space Explored (Nodes)"
    ],
    "Shortest Distance Policy": [
        f"{dist_shortest:.1f} m",
        f"{time_shortest:.1f} s",
        f"{time_shortest / 60.0:.2f} min",
        f"{(dist_shortest / time_shortest) * 3.6:.1f} km/h",
        str(len(path_shortest)),
        str(exp_shortest)
    ],
    "Fastest Travel Time Policy": [
        f"{dist_fastest:.1f} m",
        f"{time_fastest:.1f} s",
        f"{time_fastest / 60.0:.2f} min",
        f"{(dist_fastest / time_fastest) * 3.6:.1f} km/h",
        str(len(path_fastest)),
        str(exp_fastest)
    ]
})

print(comparison_df)


## 8. Geospatial Interactive Dashboard & Search Frontier Analysis

We visualize the full urban road network, the exploration wavefront of Dijkstra's algorithm, and compare both optimal trajectories on an interactive geospatial canvas using **Plotly**.


In [ ]:
def plot_osmnx_dijkstra_map(
    graph: nx.MultiDiGraph, 
    path_shortest: list[int], 
    path_fastest: list[int], 
    start_node: int, 
    goal_node: int,
    explored_order: list[int]
):
    # 1. Extract Base Street Geometry
    edge_x = []
    edge_y = []
    for u, v, data in graph.edges(data=True):
        if 'geometry' in data:
            geom = data['geometry']
            if isinstance(geom, str):
                geom = wkt.loads(geom)
            xs, ys = geom.xy
            edge_x.extend(list(xs) + [None])
            edge_y.extend(list(ys) + [None])
        else:
            x0, y0 = graph.nodes[u]['x'], graph.nodes[u]['y']
            x1, y1 = graph.nodes[v]['x'], graph.nodes[v]['y']
            edge_x.extend([x0, x1, None])
            edge_y.extend([y0, y1, None])
            
    fig = go.Figure()
    
    # 2. Road Network Base Layer
    fig.add_trace(go.Scatter(
        x=edge_x, y=edge_y,
        mode='lines',
        line=dict(width=1.2, color='#cbd5e1'),
        hoverinfo='none',
        name='OSM Road Network'
    ))
    
    # 3. Search Frontier (Explored Nodes colored by exploration sequence)
    if explored_order:
        exp_x = [graph.nodes[n]['x'] for n in explored_order]
        exp_y = [graph.nodes[n]['y'] for n in explored_order]
        exp_steps = list(range(len(explored_order)))
        fig.add_trace(go.Scatter(
            x=exp_x, y=exp_y,
            mode='markers',
            marker=dict(
                size=5,
                color=exp_steps,
                colorscale='Viridis',
                colorbar=dict(title="Dijkstra Expansion Step", x=1.02),
                opacity=0.6
            ),
            hovertext=[f"Node {n}<br>Explored Step: {i}" for i, n in enumerate(explored_order)],
            hoverinfo='text',
            name='Explored Frontier'
        ))
        
    # Helper to extract detailed continuous path coordinates
    def get_path_coords(path):
        px, py = [], []
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            edge_dict = graph[u][v]
            best_data = min(edge_dict.values(), key=lambda d: float(d.get('length', float('inf'))))
            if 'geometry' in best_data:
                geom = best_data['geometry']
                if isinstance(geom, str):
                    geom = wkt.loads(geom)
                xs, ys = geom.xy
                px.extend(list(xs))
                py.extend(list(ys))
            else:
                px.extend([graph.nodes[u]['x'], graph.nodes[v]['x']])
                py.extend([graph.nodes[u]['y'], graph.nodes[v]['y']])
        return px, py

    # 4. Shortest Distance Path (Blue Line)
    px_s, py_s = get_path_coords(path_shortest)
    fig.add_trace(go.Scatter(
        x=px_s, y=py_s,
        mode='lines',
        line=dict(width=5, color='#2563eb', dash='dash'),
        name='Shortest Distance Route'
    ))

    # 5. Fastest Travel Time Path (Crimson Line)
    px_f, py_f = get_path_coords(path_fastest)
    fig.add_trace(go.Scatter(
        x=px_f, y=py_f,
        mode='lines',
        line=dict(width=5, color='#dc2626'),
        name='Fastest Travel Time Route'
    ))

    # 6. Start & Goal Waypoints
    fig.add_trace(go.Scatter(
        x=[graph.nodes[start_node]['x']], y=[graph.nodes[start_node]['y']],
        mode='markers+text',
        marker=dict(size=16, color='#16a34a', symbol='circle'),
        text=["START (Origin)"],
        textposition="bottom center",
        name='Origin'
    ))
    fig.add_trace(go.Scatter(
        x=[graph.nodes[goal_node]['x']], y=[graph.nodes[goal_node]['y']],
        mode='markers+text',
        marker=dict(size=18, color='#f59e0b', symbol='star'),
        text=["GOAL (Destination)"],
        textposition="top center",
        name='Destination'
    ))

    fig.update_layout(
        title="Urban Mission Planning: OpenStreetMap Network & Dijkstra Route Comparison (Piedmont, CA)",
        xaxis_title="Longitude (°)",
        yaxis_title="Latitude (°)",
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)'),
        width=900,
        height=650
    )
    fig.show()

plot_osmnx_dijkstra_map(G_osm, path_shortest, path_fastest, start_node, goal_node, order_fastest)


In [ ]:
# Telemetry Metrics Comparison Bar Charts
fig_bar = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Total Distance (m)", "Estimated Travel Time (s)", "Average Speed (km/h)")
)

policies = ["Shortest Distance", "Fastest Time"]
dist_vals = [dist_shortest, dist_fastest]
time_vals = [time_shortest, time_fastest]
speed_vals = [(dist_shortest / time_shortest) * 3.6, (dist_fastest / time_fastest) * 3.6]

fig_bar.add_trace(
    go.Bar(x=policies, y=dist_vals, marker_color=["#2563eb", "#dc2626"], name="Distance"),
    row=1, col=1
)
fig_bar.add_trace(
    go.Bar(x=policies, y=time_vals, marker_color=["#2563eb", "#dc2626"], name="Time"),
    row=1, col=2
)
fig_bar.add_trace(
    go.Bar(x=policies, y=speed_vals, marker_color=["#2563eb", "#dc2626"], name="Speed"),
    row=1, col=3
)

fig_bar.update_layout(
    title="Autonomous Routing Objective Comparison (Distance vs Travel Time)",
    height=380,
    showlegend=False
)
fig_bar.show()


## 8.1. Priority Queue Mechanics & Search Space Pruning

### ❓ How Does the Min-Heap Priority Queue Select Nodes?
Dijkstra's algorithm selects the next node to expand by popping $\arg\min_{u \in \text{OpenSet}} g(u)$ from the binary min-heap (`heapq`).

- **Wavefront Propagation:** In uniform-cost search, the search expands outwards in concentric isochrones (regions of equal minimum travel time).
- **Early Termination:** As soon as `current_node == goal_node` is popped from the min-heap, the optimal cost to the destination is **guaranteed**; the algorithm **terminates immediately**.
- **Search Space Pruning:** For an intermediate destination (e.g. $1.4\text{ km}$ away), Dijkstra expands only the localized sub-graph ($pprox 33.8\%$ of nodes), leaving **$66.2\%$ of the entire city graph completely unvisited/pruned**!

Let's compute and visualize this Priority Queue early-termination effect on the map below.


In [ ]:
# Mid-Range Destination to visualize Priority Queue Early Termination
mid_goal_node = 53041443

path_mid, cost_mid, exp_mid, order_mid = osmnx_dijkstra(
    G_osm, start_node, mid_goal_node, weight='travel_time'
)

visited_set = set(order_mid)
unvisited_nodes = [n for n in G_osm.nodes if n not in visited_set]

# Active min-heap frontier: nodes adjacent to visited that were queued
frontier_nodes = set()
for u in visited_set:
    for v in G_osm.neighbors(u):
        if v not in visited_set:
            frontier_nodes.add(v)

print(f"--- Priority Queue Early-Termination Statistics ---")
print(f"• Total Network Nodes:     {len(G_osm.nodes)}")
print(f"• Explored/Settled Nodes:  {len(visited_set)} ({len(visited_set)/len(G_osm.nodes)*100:.1f}%)")
print(f"• Pruned/Unvisited Nodes:  {len(unvisited_nodes)} ({len(unvisited_nodes)/len(G_osm.nodes)*100:.1f}%)")
print(f"• Active Frontier in Heap: {len(frontier_nodes)} nodes queued waiting")

# Plotly Map showing Settled, Active Frontier, and Pruned Nodes
fig_pq = go.Figure()

# 1. Base Road Lines
edge_x, edge_y = [], []
for u, v, data in G_osm.edges(data=True):
    if 'geometry' in data:
        geom = data['geometry']
        if isinstance(geom, str):
            geom = wkt.loads(geom)
        xs, ys = geom.xy
        edge_x.extend(list(xs) + [None])
        edge_y.extend(list(ys) + [None])
    else:
        edge_x.extend([G_osm.nodes[u]['x'], G_osm.nodes[v]['x'], None])
        edge_y.extend([G_osm.nodes[u]['y'], G_osm.nodes[v]['y'], None])

fig_pq.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=1.0, color='#cbd5e1'), hoverinfo='none', name='Road Network'
))

# 2. Unvisited / Pruned Nodes (Gray)
unvis_x = [G_osm.nodes[n]['x'] for n in unvisited_nodes]
unvis_y = [G_osm.nodes[n]['y'] for n in unvisited_nodes]
fig_pq.add_trace(go.Scatter(
    x=unvis_x, y=unvis_y, mode='markers',
    marker=dict(size=4, color='#94a3b8', opacity=0.4),
    name=f'Pruned / Unvisited Nodes ({len(unvisited_nodes)} nodes = {len(unvisited_nodes)/len(G_osm.nodes)*100:.1f}%)',
    hoverinfo='none'
))

# 3. Explored / Settled Nodes (Viridis Colormap by Step)
settled_x = [G_osm.nodes[n]['x'] for n in order_mid]
settled_y = [G_osm.nodes[n]['y'] for n in order_mid]
fig_pq.add_trace(go.Scatter(
    x=settled_x, y=settled_y, mode='markers',
    marker=dict(
        size=7,
        color=list(range(len(order_mid))),
        colorscale='Viridis',
        colorbar=dict(title="Expansion Step", x=1.02),
        opacity=0.85
    ),
    hovertext=[f"Node {n}<br>Settled Step: {i}" for i, n in enumerate(order_mid)],
    hoverinfo='text',
    name=f'Settled Nodes ({len(order_mid)} nodes = {len(order_mid)/len(G_osm.nodes)*100:.1f}%)'
))

# 4. Active Min-Heap Frontier (Cyan Rings)
front_x = [G_osm.nodes[n]['x'] for n in frontier_nodes]
front_y = [G_osm.nodes[n]['y'] for n in frontier_nodes]
fig_pq.add_trace(go.Scatter(
    x=front_x, y=front_y, mode='markers',
    marker=dict(size=8, color='#06b6d4', symbol='circle-open', line=dict(width=2)),
    name=f'Active Open Set in PQ ({len(frontier_nodes)} nodes)'
))

# 5. Optimal Route
px_m, py_m = [], []
for i in range(len(path_mid) - 1):
    u, v = path_mid[i], path_mid[i+1]
    best_data = min(G_osm[u][v].values(), key=lambda d: float(d.get('travel_time', float('inf'))))
    if 'geometry' in best_data:
        geom = best_data['geometry']
        if isinstance(geom, str):
            geom = wkt.loads(geom)
        xs, ys = geom.xy
        px_m.extend(list(xs))
        py_m.extend(list(ys))
    else:
        px_m.extend([G_osm.nodes[u]['x'], G_osm.nodes[v]['x']])
        py_m.extend([G_osm.nodes[u]['y'], G_osm.nodes[v]['y']])

fig_pq.add_trace(go.Scatter(
    x=px_m, y=py_m, mode='lines',
    line=dict(width=5, color='#dc2626'),
    name=f'Optimal Route ({cost_mid:.1f} s)'
))

# 6. Start & Goal
fig_pq.add_trace(go.Scatter(
    x=[G_osm.nodes[start_node]['x']], y=[G_osm.nodes[start_node]['y']],
    mode='markers+text', marker=dict(size=16, color='#16a34a', symbol='circle'),
    text=["START"], textposition="bottom center", name='Origin'
))
fig_pq.add_trace(go.Scatter(
    x=[G_osm.nodes[mid_goal_node]['x']], y=[G_osm.nodes[mid_goal_node]['y']],
    mode='markers+text', marker=dict(size=18, color='#f59e0b', symbol='star'),
    text=["GOAL (Reached)"], textposition="top center", name='Destination'
))

fig_pq.update_layout(
    title="Priority Queue Search Pruning: Explored Subgraph vs Unvisited Network",
    xaxis_title="Longitude (°)",
    yaxis_title="Latitude (°)",
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.88)'),
    width=900,
    height=650
)
fig_pq.show()


## 8.2. Dynamic Search Wavefront Animations & Videos

Compare the two search videos below:
1. **Priority Queue Early Termination Video:** Shows the min-heap expanding a localized wavefront and **terminating immediately** once the destination is popped, pruning $66.2\%$ of the map.
2. **Full-Network Traversal Video:** Shows how Dijkstra expands across the entire urban network when the goal is situated at the distant boundary perimeter.


In [ ]:
# Video Generation Pipeline & Interactive Display
# Set generate_videos = True if you wish to re-render the animation videos from scratch
generate_videos = False

video_pq = "dijkstra_pq_early_termination.mp4"
gif_pq = "dijkstra_pq_early_termination.gif"
video_full = "dijkstra_search_evolution.mp4"
gif_full = "dijkstra_search_evolution.gif"

if generate_videos or not (os.path.exists(video_pq) and os.path.exists(video_full)):
    print("🎬 Rendering Dijkstra search animation videos (Scenario 1 & Scenario 2)...")
    try:
        from generate_dijkstra_search_video import render_scenario_video
        
        # Scenario 1: Priority Queue Early-Termination (Mid-Range Target)
        render_scenario_video(
            G_osm, start_node, mid_goal_node,
            title_scenario="Priority Queue Early-Termination (Mid-Range Target)",
            output_mp4=video_pq,
            output_gif=gif_pq,
            fps=25
        )
        
        # Scenario 2: Full-Network Traversal (Boundary Target)
        render_scenario_video(
            G_osm, start_node, goal_node,
            title_scenario="Full-Network Traversal (Boundary Target)",
            output_mp4=video_full,
            output_gif=gif_full,
            fps=25
        )
        print("✅ Dijkstra videos successfully generated!")
    except Exception as e:
        print(f"⚠️ Video generation notice: {e}")

# Display Video 1: Priority Queue Early Termination
print("🎬 Video 1: Priority Queue Early-Termination (Mid-Range Target)")
if os.path.exists(video_pq):
    html_pq = f"""
    <div style="text-align: center; margin: 15px 0;">
        <video width="850" height="480" controls autoplay loop muted style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <source src="{video_pq}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
        <p style="color: #64748b; font-size: 13px; margin-top: 6px;">
            🎬 <b>Scenario 1:</b> Dijkstra Min-Heap Early-Termination — 33.8% Explored, <b>66.2% Pruned</b>
        </p>
    </div>
    """
    display(HTML(html_pq))
elif os.path.exists(gif_pq):
    display(Image(filename=gif_pq, width=850))

# Display Video 2: Full-City Traversal
print("🎬 Video 2: Full-Network Traversal (Boundary Target)")
if os.path.exists(video_full):
    html_full = f"""
    <div style="text-align: center; margin: 15px 0;">
        <video width="850" height="480" controls autoplay loop muted style="border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.15);">
            <source src="{video_full}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
        <p style="color: #64748b; font-size: 13px; margin-top: 6px;">
            🎬 <b>Scenario 2:</b> Dijkstra Traversal to Distant City Boundary
        </p>
    </div>
    """
    display(HTML(html_full))
elif os.path.exists(gif_full):
    display(Image(filename=gif_full, width=850))


## 9. Comprehensive Verification & Unit Tests

We execute automated test assertions validating:
1. Shortest path optimality against NetworkX's reference implementation.
2. Fastest path optimality against NetworkX's reference implementation.
3. Path validity (every successive pair of nodes represents a valid directed road link).


In [ ]:
# 1. Verify Fastest Travel Time Path against NetworkX
nx_fastest_path = nx.shortest_path(G_osm, start_node, goal_node, weight='travel_time')
nx_fastest_cost = nx.shortest_path_length(G_osm, start_node, goal_node, weight='travel_time')

assert path_fastest == nx_fastest_path, f"Fastest path does not match NetworkX reference!"
assert math.isclose(time_fastest, nx_fastest_cost, rel_tol=1e-4), (
    f"Computed time {time_fastest:.2f}s does not match NetworkX {nx_fastest_cost:.2f}s!"
)

# 2. Verify Shortest Distance Path against NetworkX
nx_shortest_path = nx.shortest_path(G_osm, start_node, goal_node, weight='length')
nx_shortest_cost = nx.shortest_path_length(G_osm, start_node, goal_node, weight='length')

assert path_shortest == nx_shortest_path, f"Shortest path does not match NetworkX reference!"
assert math.isclose(dist_shortest, nx_shortest_cost, rel_tol=1e-4), (
    f"Computed distance {dist_shortest:.2f}m does not match NetworkX {nx_shortest_cost:.2f}m!"
)

# 3. Verify Edge Feasibility along Path
for p_name, p_nodes in [("Shortest Path", path_shortest), ("Fastest Path", path_fastest)]:
    for i in range(len(p_nodes) - 1):
        u, v = p_nodes[i], p_nodes[i+1]
        assert G_osm.has_edge(u, v), f"Invalid edge transition from {u} to {v} in {p_name}!"

print("✅ All Real-World OSMnx Dijkstra Mission Planning Tests Passed Successfully!")
